# Notebook 01 · Exploración inicial de datasets (Paso 1.3)

**Objetivo:** Examinar los 4 datasets crudos en español para entender su forma, sus columnas, su distribución de etiquetas y su cobertura LATAM **antes** de limpiar (Paso 1.4) y unificar (Paso 1.5).

Datasets:
1. **HatEval 2019** — Twitter, anotación binaria + dimensiones (HS, TR, AG).
2. **DETOXIS 2021** — comentarios de noticias, 20 dimensiones.
3. **HaterNet** — Twitter, anotación binaria simple.
4. **Chilean Dataset** — Twitter chileno, 17 dimensiones.

Salidas que produce este notebook:
- `data/reports_qc/exploracion_inicial.md` (reporte ejecutivo).
- `data/reports_qc/exploracion_inicial.json` (métricas crudas).
- `data/reports_qc/figuras/*.png` (4 figuras).

> Las funciones que se usan están en `scripts/exploracion_inicial.py`. Este notebook llama a las mismas, pero permite ejecutar paso a paso e inspeccionar los DataFrames intermedios.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

# Asegurar que `scripts/` esté en el path para reusar funciones.
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT / 'scripts') not in sys.path:
    sys.path.insert(0, str(ROOT / 'scripts'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from exploracion_inicial import (
    cargar_hateval, cargar_detoxis, cargar_haternet, cargar_chilean,
    explorar_dataset, figura_volumen, figura_distribucion_clases,
    figura_longitud_texto, figura_seeds_latam, render_md,
    SEEDS_LATAM, REPORTS, FIGS,
)

pd.set_option('display.max_colwidth', 120)
print('Raíz del proyecto:', ROOT)

## 2. Cargar los 4 datasets

In [ ]:
df_hateval = cargar_hateval()
df_detoxis = cargar_detoxis()
df_haternet = cargar_haternet()
df_chilean = cargar_chilean()

print(f'HatEval (ES): {df_hateval.shape}')
print(f'DETOXIS    : {df_detoxis.shape}')
print(f'HaterNet   : {df_haternet.shape}')
print(f'Chilean    : {df_chilean.shape}')

### 2.1 HatEval

In [ ]:
df_hateval.head(3)

In [ ]:
print('Columnas:', df_hateval.columns.tolist())
print('Dtypes:'); print(df_hateval.dtypes)
print()
print('Distribución HS:'); print(df_hateval['HS'].value_counts())
print()
print('Distribución TR:'); print(df_hateval['TR'].value_counts())
print()
print('Distribución AG:'); print(df_hateval['AG'].value_counts())

### 2.2 DETOXIS

In [ ]:
df_detoxis.head(3)

In [ ]:
print('Columnas:', df_detoxis.columns.tolist())
print()
print('Distribución toxicity (binaria):')
print(df_detoxis['toxicity'].value_counts())
print()
print('Distribución toxicity_level (ordinal):')
print(df_detoxis['toxicity_level'].value_counts().sort_index())

### 2.3 HaterNet

In [ ]:
df_haternet.head(3)

In [ ]:
print('Distribución label:')
print(df_haternet['label'].value_counts())
print()
print('Proporción odio:', df_haternet['label'].mean().round(4))

### 2.4 Chilean Dataset

In [ ]:
df_chilean[['caso', 'tweet a etiquetar', 'hate speech/estereotipo']].head(3)

In [ ]:
print('Filas totales:', len(df_chilean))
print('Filas con texto válido:', df_chilean['tweet a etiquetar'].notna().sum())
print()
print('Distribución hate speech/estereotipo:')
print(df_chilean['hate speech/estereotipo'].value_counts(dropna=False))

## 3. Estadísticas resumidas (todos)

In [ ]:
resumenes = {
    'HatEval': explorar_dataset(
        'HatEval', df_hateval, 'text',
        ['HS', 'TR', 'AG']
    ),
    'DETOXIS': explorar_dataset(
        'DETOXIS', df_detoxis, 'comment',
        ['toxicity', 'aggressiveness', 'insult', 'stereotype',
         'target_person', 'target_group']
    ),
    'HaterNet': explorar_dataset(
        'HaterNet', df_haternet, 'text',
        ['label']
    ),
    'Chilean': explorar_dataset(
        'Chilean', df_chilean, 'tweet a etiquetar',
        ['hate speech/estereotipo', 'insulto/sobrenombre',
         'grosería c/int.', 'sarcasmo/ironía/burla', 'mención migración']
    ),
}

tabla = []
for ds, d in resumenes.items():
    if d.get('vacio'):
        tabla.append({'dataset': ds, 'filas': 0, 'longitud_p95_tokens': 0,
                      'pct_seeds_latam': 0})
        continue
    tabla.append({
        'dataset': ds,
        'filas': d['shape'][0],
        'longitud_p95_tokens': d['longitud']['tokens_p95'],
        'pct_seeds_latam': round(d['seeds_latam_prop'] * 100, 2),
    })
pd.DataFrame(tabla)

## 4. Figuras

In [ ]:
rutas = {
    'volumen':  figura_volumen(resumenes),
    'clases':   figura_distribucion_clases(resumenes),
    'longitud': figura_longitud_texto(resumenes),
    'seeds':    figura_seeds_latam(resumenes),
}
for k, v in rutas.items():
    print(f'{k:9s} -> {v}')

Mostrar las imágenes inline en el notebook:

In [ ]:
from IPython.display import Image, display
for k, v in rutas.items():
    print(f'\n— {k} —')
    display(Image(filename=str(v)))

## 5. Reporte Markdown + JSON

In [ ]:
import json
md = render_md(resumenes, rutas)
(REPORTS / 'exploracion_inicial.md').write_text(md, encoding='utf-8')
(REPORTS / 'exploracion_inicial.json').write_text(
    json.dumps(resumenes, ensure_ascii=False, indent=2, default=str),
    encoding='utf-8'
)
print('Reportes generados en:', REPORTS)

## 6. Hallazgos

1. **Volumen** (filas tras carga inicial, HatEval filtrado a ES) → ver figura `volumen_datasets.png`.
2. **Etiquetas heterogéneas:** cada dataset usa una columna distinta como etiqueta principal de hate. El Paso 1.5 las mapeará a `etiqueta ∈ {0, 1}` con la regla:
   - HatEval → `HS == 1`
   - DETOXIS → `toxicity_level >= 2` (o `toxicity == 1`)
   - HaterNet → `label == 1`
   - Chilean → `hate speech/estereotipo == 1`
3. **Modismos LATAM:** `Chilean` concentra la mayor proporción de seeds, lo que valida su importancia para H3.
4. **Longitud de texto:** los comentarios de DETOXIS son significativamente más largos que los tweets, lo que afectará el truncado a 128/256 tokens en el fine-tuning de BETO.

## 7. Próximo paso

→ **Paso 1.4** · Crear `src/data/clean.py` con `normalizar()` y aplicarla en `notebooks/02_unificacion.ipynb`.